# Muestreo de máscaras de I-JEPA: cobertura de la rejilla

Cuaderno de lectura de la medición `ijepa/` del repositorio [ManPlaNet-datos](https://github.com/mmunozpl/ManPlaNet-datos). Respalda el artículo [predecir-sin-dibujar](https://manpla.net/posts/predecir-sin-dibujar/). Carga el fichero de al lado —o lo descarga del repositorio si se ejecuta fuera de él—, muestra la ficha de procedencia y dibuja una figura con matplotlib a secas. Solo lee; no vuelve a tomar la instantánea: para eso está `generar.py`.

*Reading notebook for this measurement: loads the file next to it, prints the provenance record and draws one figure. Column names are in Spanish; `GLOSARIO.md` gives the English form.*

In [ ]:
import io, json, urllib.request
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RAW = "https://raw.githubusercontent.com/mmunozpl/ManPlaNet-datos/main/ijepa/"

def leer(nombre, **kw):
    """el fichero de al lado si existe; si no, el del repositorio."""
    p = Path(nombre)
    if p.exists():
        return pd.read_csv(p, **kw)
    return pd.read_csv(RAW + nombre, **kw)

def texto(nombre):
    p = Path(nombre)
    if p.exists():
        return p.read_text(encoding="utf-8")
    with urllib.request.urlopen(RAW + nombre, timeout=30) as r:
        return r.read().decode("utf-8")


## Ficha de procedencia

In [ ]:
print(texto("resumen.json"))

## El dato

In [ ]:
cob = leer("cobertura.csv")
res = json.loads(texto("resumen.json"))
print("imágenes simuladas:", res["imagenes"], "· contexto medio (bruto):", round(res["contexto_bruto"]["media"], 1), "parches")
cob.describe().round(1)

## Una figura

In [ ]:
import numpy as np
lado = cob.fila.max() + 1
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, col, t in ((a, "veces_contexto", "veces como contexto"), (b, "veces_objetivo", "veces como objetivo")):
    m = cob.pivot(index="fila", columns="columna", values=col).values
    im = ax.imshow(m, cmap="viridis"); ax.set_title(t); fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()